Obtaining id's and compositions of materials from Materials Project

In [4]:
import numpy as np 
import pandas as pd 
from tqdm import tqdm 
import mendeleev 
from mp_api.client import MPRester 

# retrieving materials in Materials Project with naturally-occurring elements

elements = mendeleev.get_all_elements()
element_symbols = [element.symbol for element in elements]
natural_element_symbols = element_symbols[:92]

# e.g. element_types = natural_element_symbols
def fetch_Materials_Project(element_types, MP_api_key):
    all_ids = []
    all_formulas = []

    for element in tqdm(element_types):
        with MPRester(use_document_model = False, api_key = MP_api_key) as mpr:
            try:
                docs = mpr.materials.summary.search(
                    elements = [element], 
                    fields = ["material_id", "formula_pretty"]
                )
                for doc in docs:
                    id = doc["material_id"]
                    formula = doc["formula_pretty"]

                    if id not in all_ids:
                        all_ids.append(id)
                        all_formulas.append(formula)
            except Exception as e:
                continue 
            
    all_ids = np.array(all_ids)
    all_formulas = np.array(all_formulas)
    return all_ids, all_formulas

Generating machine learning representations from chemical compositions

In [13]:
from matminer.featurizers.conversions import StrToComposition 
from matminer.featurizers.composition import ElementProperty 
from matminer.featurizers.composition.element import TMetalFraction 
from matminer.featurizers.composition.element import Stoichiometry 
from matminer.featurizers.composition.element import BandCenter 
from matminer.featurizers.composition.orbital import AtomicOrbitals 
from matminer.featurizers.composition.orbital import ValenceOrbital 

# input Panda dataframe containing strings of chemical compositions
def generate_Matminer_features(compositions_df, composition_column_name, array_of_features_of_interest):
    composition = StrToComposition(target_col_id = "composition")
    data = composition.featurize_dataframe(compositions_df, col_id = composition_column_name)
    featurizer = ElementProperty.from_preset("magpie")
    transition_metal_fraction = TMetalFraction() 
    stoichiometry = Stoichiometry() 
    band_center = BandCenter()
    atomic_orbitals = AtomicOrbitals() 
    valence_orbitals = ValenceOrbital() 

    featurized_data0 = featurizer.featurize_dataframe(data, col_id = "composition")
    featurized_data_transition_metal_fraction = transition_metal_fraction.featurize_dataframe(
        featurized_data0, col_id = "composition"
    )
    featurized_data_stoichiometry = stoichiometry.featurize_dataframe(
        featurized_data_transition_metal_fraction, col_id = "composition"
    )
    featurized_data_band_center = band_center.featurize_dataframe(
        featurized_data_stoichiometry, col_id = "composition", ignore_errors = True
    )
    featurized_data_atomic_orbitals = atomic_orbitals.featurize_dataframe(
        featurized_data_band_center, col_id = "composition", ignore_errors = True
    )
    featurized_data_valence_orbitals = valence_orbitals.featurize_dataframe(
        featurized_data_atomic_orbitals, col_id = "composition", ignore_errors = True
    )
    
    featurized_data = featurized_data_valence_orbitals.drop("composition", axis = 1)
    featurized_data = featurized_data.dropna(axis = 1)
    featurized_data = featurized_data[array_of_features_of_interest]

    return featurized_data

Similarity-Based Machine Learning

In [17]:
from sklearn.linear_model import RidgeCV, Ridge 
from sklearn.neighbors import KNeighborsRegressor 

def ridge_regression_prediction_and_coefficients(train_features, train_labels, test_features):
    alphas_range = np.linspace(0.01, 50, 500)
    ridgecv = RidgeCV(alphas = alphas_range)
    ridgecv.fit(np.array(train_features), train_labels)
    alpha = ridgecv.alpha_ 

    ridge = Ridge(alpha = alpha, solver = "cholesky")
    ridge.fit(np.array(train_features), train_labels)
    ridge_prediction = ridge.predict(np.array(test_features))[0]
    ridge_prediction = np.abs(ridge_prediction)
    ridge_coefficients = ridge.coef_ 

    return ridge_prediction, ridge_coefficients 

def similarity_based_ridge_regression(train_features, train_labels, test_features, n_neighbors):
    knn_regressor = KNeighborsRegressor(n_neighbors = n_neighbors, n_jobs = -1)
    knn_regressor.fit(train_features, train_labels)
    knn_indices = knn_regressor.kneighbors(test_features)[1][0]
    test_prediction, ridge_coefficients = ridge_regression_prediction_and_coefficients(
        train_features.iloc[knn_indices], 
        train_labels[knn_indices], 
        test_features
    )
    
    return test_prediction, ridge_coefficients